# Notebook 05 — Export et validation du paquet de déploiement (Rebuild 2026)

| | |
|---|---|
| **Projet** | Application Intelligente de Scoring Client |
| **Entreprise** | Orus Services — filiale Salafin, Bank of Africa |
| **Auteure** | Fatima Zahra Ait Lamine |
| **Année** | PFE 2025–2026 (rebuild du pipeline ML) |

---

## Objectif

Valider le **paquet de déploiement** du modèle de production (XGBoost contraint + calibration
isotonique + seuil optimisé, sélection phase 3, évaluation officielle phase 4) :

1. Inventaire des artefacts et de leur rôle pour le service FastAPI.
2. Contrôles de cohérence (préprocesseur, ordre des features, contraintes, métadonnées).
3. Démonstration de bout en bout : profils clients bruts → PD calibrée → décision.
4. **Reproduction dans un processus Python vierge** (sous-processus indépendant).
5. Contrat d'intégration pour `main.py` (phase 7).

> L'explicabilité SHAP du modèle de production fait l'objet de la **phase suivante** — elle
> n'est volontairement pas traitée ici.

## 1. Inventaire du paquet de déploiement

In [1]:
import sys, os, pickle, subprocess, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
sys.path.append(os.path.abspath('../src'))
from preprocessing import (clean_gmsc, FEATURE_COLS, RAW_FEATURES, MONOTONE_CONSTRAINTS,
                           PREPROCESSING_VERSION, ScoringModel)
from scipy.special import logit

MODELS_PATH = '../models/'
def load(name):
    with open(MODELS_PATH + name, 'rb') as f: return pickle.load(f)

ROLES = {
    'model_final.pkl':                  'PRODUCTION — ScoringModel (préprocesseur fitté + XGBoost contraint)',
    'calibrator.pkl':                   'PRODUCTION — calibreur isotonique (PD brute → PD calibrée)',
    'decision_threshold.pkl':           'PRODUCTION — seuil de décision optimisé (F1, OOF train)',
    'preprocessor.pkl':                 'PRODUCTION — FittedPreprocessor autonome (référence/contrôles)',
    'feature_cols.pkl':                 'PRODUCTION — ordre canonique des 12 features',
    'metadata_final.pkl':               'PRODUCTION — traçabilité (config, seed, versions, métriques CV+test)',
    'monotone_constraints.pkl':         'Documentation — contraintes vérifiées (phase 2)',
    'sample_check.pkl':                 'Qualité — échantillon de non-régression inter-sessions',
    'model_random_forest.pkl':          'Comparaison — gagnant quantitatif CV (conservé, non déployé)',
    'calibrator_random_forest.pkl':     'Comparaison — calibreur Platt du RF',
    'model_logistic_regression.pkl':    'Comparaison — référence linéaire',
    'calibrator_logistic_regression.pkl': 'Comparaison — calibreur isotonique de la LR',
    'cv_results_phase3.csv':            'Résultats — tableau CV complet (phase 3)',
    'calibration_comparison_phase3.csv': 'Résultats — Brier/LogLoss/ECE par méthode de calibration',
    'oof_predictions.csv':              'Résultats — prédictions OOF (réutilisées au mapping de score)',
    'test_metrics_phase4.csv':          'Résultats — métriques officielles du test (phase 4)',
    'calibration_table_test_phase4.csv': 'Résultats — table de calibration test par décile',
}
print(f"{'artefact':<38} {'taille':>9}  rôle")
print('-' * 120)
manquants = []
for name, role in ROLES.items():
    p = MODELS_PATH + name
    if os.path.exists(p):
        print(f"{name:<38} {os.path.getsize(p)/1024:>8.1f}K  {role}")
    else:
        manquants.append(name)
assert not manquants, f'artefacts manquants : {manquants}'
print(f"\n✓ {len(ROLES)} artefacts présents")

artefact                                  taille  rôle
------------------------------------------------------------------------------------------------------------------------
model_final.pkl                          1348.9K  PRODUCTION — ScoringModel (préprocesseur fitté + XGBoost contraint)
calibrator.pkl                              2.9K  PRODUCTION — calibreur isotonique (PD brute → PD calibrée)
decision_threshold.pkl                      0.0K  PRODUCTION — seuil de décision optimisé (F1, OOF train)
preprocessor.pkl                            0.6K  PRODUCTION — FittedPreprocessor autonome (référence/contrôles)
feature_cols.pkl                            0.3K  PRODUCTION — ordre canonique des 12 features
metadata_final.pkl                          2.9K  PRODUCTION — traçabilité (config, seed, versions, métriques CV+test)
monotone_constraints.pkl                    0.3K  Documentation — contraintes vérifiées (phase 2)
sample_check.pkl                            1.9K  Qualité — échant

## 2. Contrôles de cohérence

In [2]:
model      = load('model_final.pkl')
calibrator = load('calibrator.pkl')
threshold  = load('decision_threshold.pkl')
metadata   = load('metadata_final.pkl')

# a) le modèle de production est bien le ScoringModel XGBoost attendu
assert isinstance(model, ScoringModel) and type(model.clf).__name__ == 'XGBClassifier'
assert metadata['modele_production'] == 'XGBoost' and calibrator['method'] == 'isotonic'

# b) préprocesseur du modèle == artefact autonome (mêmes caps, mêmes médianes)
prep_ref = load('preprocessor.pkl')
assert (model.prep.income_cap_ == prep_ref.income_cap_ and
        model.prep.debtratio_cap_ == prep_ref.debtratio_cap_ and
        model.prep.dependents_cap_ == prep_ref.dependents_cap_ and
        all(abs(model.prep.medians_[k] - prep_ref.medians_[k]) < 1e-12 for k in prep_ref.medians_))

# c) ordre des features : artefact == module == booster
assert load('feature_cols.pkl') == FEATURE_COLS == list(model.clf.get_booster().feature_names)

# d) contraintes monotones : artefact == module == booster réellement contraint
assert load('monotone_constraints.pkl') == MONOTONE_CONSTRAINTS
booster_mono = model.clf.get_params()['monotone_constraints']
assert tuple(booster_mono) == tuple(MONOTONE_CONSTRAINTS[c] for c in FEATURE_COLS)

# e) seuil et versions
assert 0 < threshold < 1 and abs(threshold - metadata['seuil_decision']) < 1e-12
print('✓ Modèle production = ScoringModel(FittedPreprocessor + XGBClassifier contraint)')
print(f"✓ Préprocesseur cohérent (caps : revenu {model.prep.income_cap_:,.0f} $, DR {model.prep.debtratio_cap_:.3f}, dep {model.prep.dependents_cap_:.0f})")
print(f"✓ 12 features dans l'ordre canonique ; contraintes monotones actives dans le booster")
print(f"✓ Seuil : {threshold:.4f} | calibration : {calibrator['method']} | prep : {metadata['preprocessing_version']}")
print(f"✓ Versions d'entraînement : {metadata['versions']}")
print(f"✓ Métriques officielles test (phase 4) : AUC {metadata['phase4_test_metrics']['ROC-AUC']:.4f}, "
      f"Brier {metadata['phase4_test_metrics']['Brier (cal.)']:.4f}, F1 {metadata['phase4_test_metrics']['F1 @seuil']:.4f}")

✓ Modèle production = ScoringModel(FittedPreprocessor + XGBClassifier contraint)
✓ Préprocesseur cohérent (caps : revenu 35,000 $, DR 2.016, dep 6)
✓ 12 features dans l'ordre canonique ; contraintes monotones actives dans le booster
✓ Seuil : 0.2227 | calibration : isotonic | prep : 2.0-rebuild-2026-07
✓ Versions d'entraînement : {'python': '3.12.7', 'numpy': '1.26.4', 'pandas': '2.2.3', 'scikit-learn': '1.6.1', 'xgboost': '2.1.3'}
✓ Métriques officielles test (phase 4) : AUC 0.8627, Brier 0.0489, F1 0.4512


## 3. Démonstration de bout en bout — du dossier client brut à la décision

Trois profils synthétiques (les 10 variables brutes du bureau de crédit, telles que le
backend les enverra). Chaîne exécutée : `clean_gmsc` → `ScoringModel.predict_proba` →
calibreur → seuil.

In [3]:
def apply_calibrator(cal, p):
    if cal['method'] == 'platt':
        return cal['model'].predict_proba(logit(np.clip(p, 1e-6, 1-1e-6)).reshape(-1, 1))[:, 1]
    if cal['method'] == 'isotonic':
        return cal['model'].predict(p)
    return p

profils = pd.DataFrame([
    # profil sain : faible utilisation, aucun retard, revenu confortable
    dict(zip(RAW_FEATURES, [0.05, 45, 0, 0.25, 6500, 6, 0, 1, 0, 1])),
    # profil intermédiaire : utilisation élevée, 1 retard 30-59j
    dict(zip(RAW_FEATURES, [0.75, 33, 1, 0.45, 3200, 4, 0, 0, 0, 2])),
    # profil risqué : cartes saturées, retards graves répétés, revenu inconnu
    dict(zip(RAW_FEATURES, [0.98, 28, 2, np.nan, np.nan, 3, 2, 0, 1, 0])),
], index=['sain', 'intermédiaire', 'risqué'])

X_demo = clean_gmsc(profils)[FEATURE_COLS]
pd_brute = model.predict_proba(X_demo)[:, 1]
pd_cal   = apply_calibrator(calibrator, pd_brute)
demo = pd.DataFrame({
    'PD brute': (pd_brute * 100).round(1),
    'PD calibrée (%)': (pd_cal * 100).round(1),
    f'décision (seuil {threshold:.3f})': np.where(pd_cal >= threshold, 'RISQUE ÉLEVÉ — revue', 'accepté'),
}, index=profils.index)
print(demo.to_string())
print('\nOrdre attendu respecté : sain < intermédiaire < risqué.')
print('Le profil « risqué » illustre les NaN natifs : revenu et DebtRatio inconnus → flag income_missing.')
assert pd_cal[0] < pd_cal[1] < pd_cal[2]

                PD brute  PD calibrée (%) décision (seuil 0.223)
sain            7.900000              0.9                accepté
intermédiaire  69.199997             14.6                accepté
risqué         97.599998             54.8   RISQUE ÉLEVÉ — revue

Ordre attendu respecté : sain < intermédiaire < risqué.
Le profil « risqué » illustre les NaN natifs : revenu et DebtRatio inconnus → flag income_missing.


## 4. Reproduction dans un processus Python vierge

Un **sous-processus indépendant** (nouvel interpréteur, aucun objet hérité du notebook)
recharge les artefacts et doit reproduire les prédictions de référence de
`sample_check.pkl` à 10⁻¹⁰ près.

In [4]:
script = '''
import sys, pickle, numpy as np
sys.path.insert(0, sys.argv[1])
from preprocessing import ScoringModel
from scipy.special import logit
M = sys.argv[2]
def load(n):
    with open(M + n, "rb") as f: return pickle.load(f)
model, cal, chk = load("model_final.pkl"), load("calibrator.pkl"), load("sample_check.pkl")
p = model.predict_proba(chk["sample_raw"])[:, 1]
pc = cal["model"].predict(p) if cal["method"] == "isotonic" else p
assert np.allclose(p, chk["p_raw_expected"], atol=1e-10)
assert np.allclose(pc, chk["p_cal_expected"], atol=1e-10)
assert float(load("decision_threshold.pkl")) == chk["threshold"]
print("FRESH-SESSION OK :", np.round(pc, 5))
'''
r = subprocess.run([sys.executable, '-c', script,
                    os.path.abspath('../src'), os.path.abspath('../models') + os.sep],
                   capture_output=True, text=True)
print(r.stdout.strip())
assert r.returncode == 0, r.stderr
print('✓ Un interpréteur vierge recharge le paquet et reproduit les prédictions à 1e-10.')

FRESH-SESSION OK : [0.04014 0.00518 0.02784 0.08846 0.01866]
✓ Un interpréteur vierge recharge le paquet et reproduit les prédictions à 1e-10.


## 5. Contrat d'intégration pour le service FastAPI (phase 7)

| Élément | Spécification |
|---|---|
| Artefacts à charger au démarrage | `model_final.pkl`, `calibrator.pkl`, `decision_threshold.pkl`, `feature_cols.pkl`, `metadata_final.pkl` (+ `src/preprocessing.py` importé **avant** le dépickling) |
| Entrée API | Le payload 14 champs actuel d'`IaService.java` est **conservé** (aucun changement backend) : le service utilise les 10 variables brutes, recalcule les 2 flags via `clean_gmsc`, et **ignore** les 4 features composites héritées (`charges_mensuelles`, `score_retards`, `historique_financier`, `nb_credits_total`) |
| Chaîne de scoring | payload → DataFrame 10 colonnes → `clean_gmsc` → `model.predict_proba` → calibreur → PD calibrée |
| Score applicatif | PD calibrée × 100 (interprétable : « 12 » = 12 % de probabilité de défaut) |
| Décision binaire | PD calibrée ≥ 0.2227 → dossier à risque (seuil F1-optimal, généralisation vérifiée phase 4 : perte 0.0008) |
| Niveaux FAIBLE/MOYEN/ÉLEVÉ | À définir en phase 6 (mapping de score sur PD calibrée) — **pas** de re-bucketing arbitraire |
| Explications | SHAP sur `model.clf` (TreeExplainer) — phase suivante |

> `main.py` actuel référence encore les artefacts de l'ancien pipeline (`model_B1.pkl`, etc.) :
> sa réécriture est précisément l'objet de la **phase 7** du plan et n'est pas anticipée ici.

## 6. Synthèse

| Contrôle | Résultat |
|---|---|
| 17 artefacts présents et typés | ✓ |
| Préprocesseur modèle ≡ artefact autonome | ✓ |
| Ordre des features ≡ module ≡ booster | ✓ |
| Contraintes monotones actives dans le booster | ✓ |
| Bout-en-bout : 3 profils bruts → PD calibrée → décision, ordre cohérent | ✓ |
| Reproduction en interpréteur vierge (1e-10) | ✓ |

⏸ **STOP — le paquet de déploiement est validé.** Phases restantes du plan :
explicabilité SHAP du modèle de production, mapping de score (niveaux applicatifs),
réécriture de `main.py` (phase 7) et test d'intégration backend (phase 8).